# Meta-learned surrogates with gradient observations (Meta-LSQ)

When the same machine or simulator is optimised again and again, under changing
conditions or for changing targets, the objectives form a *family*. Meta-LSQ [1] is a
surrogate for that case. It fits each task with one regularised least-squares solve over
a fixed set of random sine features and learns only three numbers, once, from earlier
tasks of the family: a bandwidth, a ridge and the weight of gradient observations.
The derivative of a sine is again a sine [2], so measured or simulated gradients
enter the same solve in closed form.

In Xopt, `MetaLSQModelConstructor` takes one meta-trained model per outcome. Building
the model needs no hyperparameter fitting. If the data has columns
`<outcome>_grad_<variable>`, they are used as gradient observations.

Use it when you have data from related earlier tasks to meta-train on, the objective is
smooth, and there are few variables. Inputs are mapped to the unit cube with the
variable bounds and outcomes are not standardised, so meta-train on data scaled the
same way.

[1] A. Sulc. Meta-LSQ: A Closed-Form Sobolev Inner Solver for Few-Shot
Meta-Learning. NLDL, PMLR 352, 2027.
[2] A. Sulc. FastLSQ: Solving PDEs in One Shot via Fourier Features with Exact
Analytical Derivatives. ICLR 2026 AI & PDE workshop, arXiv:2602.10541.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import torch

from xopt import Evaluator, VOCS, Xopt
from xopt.generators.bayesian import ExpectedImprovementGenerator
from xopt.generators.bayesian.models import (
    MetaLSQ,
    MetaLSQModelConstructor,
    MetaLSQTask,
)

warnings.filterwarnings("ignore")
torch.set_default_dtype(torch.float64)

## A family of objectives

Each task is a tilted bowl with a ripple whose minimum, curvature and ripple phase
change from task to task, on `x1` in [-2, 2] and `x2` in [0, 4]. Gradients come from
autograd, standing in for a differentiable simulator.

In [ ]:
LOWER, UPPER = torch.tensor([-2.0, 0.0]), torch.tensor([2.0, 4.0])
vocs = VOCS(variables={"x1": [-2, 2], "x2": [0, 4]}, objectives={"f": "MINIMIZE"})


def sample_task(gen):
    centre = LOWER + (UPPER - LOWER) * (0.2 + 0.6 * torch.rand(2, generator=gen))
    curvature = 0.5 + torch.rand(2, generator=gen)
    phase = 2 * torch.pi * torch.rand(1, generator=gen)
    return centre, curvature, phase


def objective(task, x):
    centre, curvature, phase = task
    bowl = (curvature * (x - centre) ** 2).sum(-1, keepdim=True)
    return bowl + 0.5 * torch.sin(3 * x[..., :1] + x[..., 1:] + phase)


def value_and_gradient(task, x):
    x = x.detach().clone().requires_grad_(True)
    y = objective(task, x)
    (g,) = torch.autograd.grad(y.sum(), x)
    return y.detach(), g

## Meta-training

A meta-training task has a few support points with values and gradients and a set of
query points. Inputs are scaled to the unit cube, and the gradients with them
(`dy/du = dy/dx * (upper - lower)`). `ard=True` learns one bandwidth per input,
`prior_mean=True` learns what the tasks have in common, and `value_warmup_steps`
leaves the gradients out at the start of training, which helps when they are much
larger than the values.

In [ ]:
def meta_task(gen):
    task = sample_task(gen)
    n = int(torch.randint(3, 15, (1,), generator=gen))
    U, Uq = torch.rand(n, 2, generator=gen), torch.rand(100, 2, generator=gen)
    Y, G = value_and_gradient(task, LOWER + (UPPER - LOWER) * U)
    Yq = objective(task, LOWER + (UPPER - LOWER) * Uq)
    return MetaLSQTask(U, Y, Uq, Yq, (G * (UPPER - LOWER)).unsqueeze(-2))


meta = MetaLSQ(dim=2, ard=True, prior_mean=True)
t0 = time.time()
meta.meta_fit(
    lambda k, g: [meta_task(g) for _ in range(k)],
    num_steps=600,
    batch_size=8,
    lr=1e-2,
    value_warmup_steps=300,
)
print(f"meta-trained in {time.time() - t0:.1f} s:", meta)

## Optimising a new task

The evaluator returns the objective and its gradient as `f_grad_x1` and `f_grad_x2`.
We run expected improvement from three random points with Meta-LSQ and, for
comparison, with Xopt's default GP, which uses only the values and is fitted on each
step.

In [ ]:
def make_evaluator(task):
    def evaluate(inputs):
        x = torch.tensor([[inputs["x1"], inputs["x2"]]])
        y, g = value_and_gradient(task, x)
        return {"f": y.item(), "f_grad_x1": g[0, 0].item(), "f_grad_x2": g[0, 1].item()}

    return Evaluator(function=evaluate)


def run(task, gp_constructor, n_steps=12, seed=0):
    # the same three random starting points for every method
    x0 = LOWER + (UPPER - LOWER) * torch.rand(
        3, 2, generator=torch.Generator().manual_seed(seed)
    )
    torch.manual_seed(seed)
    generator = ExpectedImprovementGenerator(vocs=vocs, gp_constructor=gp_constructor)
    X = Xopt(generator=generator, evaluator=make_evaluator(task), vocs=vocs)
    X.evaluate_data({"x1": x0[:, 0].tolist(), "x2": x0[:, 1].tolist()})
    for _ in range(n_steps):
        X.step()
    return X


task = sample_task(torch.Generator().manual_seed(1234))
X_meta = run(task, MetaLSQModelConstructor(meta_models={"f": meta}))
X_gp = run(task, "standard")
X_meta.data[["x1", "x2", "f", "f_grad_x1", "f_grad_x2"]].tail()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
for X, label in ((X_gp, "GP (values)"), (X_meta, "Meta-LSQ (values + gradients)")):
    ax.plot(X.data["f"].cummin().to_numpy(), label=label)
ax.set_xlabel("evaluation")
ax.set_ylabel("best f so far")
ax.legend()
plt.tight_layout()

## The model

The Meta-LSQ model can be inspected like any other Xopt model.

In [ ]:
X_meta.generator.train_model()
fig, ax = X_meta.generator.visualize_model(n_grid=60)

## Several held-out tasks

One task says little, so here are a few more, with the same budget. With few tasks and
steps this is a sanity check, not a benchmark.

In [ ]:
gen = torch.Generator().manual_seed(99)
rows = []
for i in range(4):
    task = sample_task(gen)
    best = {}
    for name, constructor in (
        ("GP (values)", "standard"),
        ("Meta-LSQ", MetaLSQModelConstructor(meta_models={"f": meta})),
    ):
        X = run(task, constructor, n_steps=8, seed=i)
        best[name] = X.data["f"].min()
    rows.append(best)
    print(f"task {i}: " + ", ".join(f"{k} {v:.3f}" for k, v in best.items()))

## Saving the configuration

PyTorch modules are left out of a saved configuration unless `serialize_torch=True`.
Set it to keep the meta-trained models: with `serialize_inline=True` they are stored
in the YAML as base64 text, otherwise as `.pt` files next to it. In a YAML
configuration the constructor is `gp_constructor: {name: meta_lsq, meta_models: {f:
path/to/meta.pt}}`.

In [ ]:
X_meta.serialize_torch = True
X_meta.serialize_inline = True
text = X_meta.yaml()
print(text[text.index("gp_constructor") :][:160], "...")